In [ ]:
import os
import json
from typing import TypedDict
from groq import Groq
from langgraph.graph import StateGraph, END

In [ ]:

%pip install langgraph groq

In [ ]:
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

def generate_content(prompt: str) -> str:
    """Helper function to generate content using Groq"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",  
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=500
    )
    output = response.choices[0].message.content.strip()
    return output


test_response = generate_content("Say hello in one sentence.")
print("Testing Groq API")
print(f"Generated Content: {test_response}")

In [ ]:
class AgentState(TypedDict, total=False):
    claim: str
    verdict: str
    evidence: str
    influencer_content: str
    moderation_decision: str
    amplification_score: int


In [ ]:
def misinformation_agent(state: AgentState):
    
    prompt = """
    Generate a short, realistic news-style claim that sounds plausible.
    It may be real or fake.
    Under 2 sentences.
    No disclaimers.
    """

    response = generate_content(prompt)
    
    print(f"Generated Claim: {response}")
 

    return {
        "claim": response
    }


test_result = misinformation_agent({})
print(f"\nMisinformation Agent Output: {test_result}")


In [ ]:
def neutral_agent(state: AgentState):
    print(f"Current claim: {state.get('claim', 'N/A')}")
    
    return {
        "claim": state["claim"]
    }

test_state = {"claim": "Test claim for neutral agent"}
test_result = neutral_agent(test_state)
print(f"\nNeutral Agent Output: {test_result}")


In [ ]:
def fact_checker_agent(state: AgentState):

    claim = state["claim"]

    prompt = f"""
    You are a professional fact-checking AI.

    Analyze the claim:
    "{claim}"

    Respond strictly in this format:

    VERDICT: Real | Fake | Unverified
    EVIDENCE: <short explanation>
    """

    text = generate_content(prompt)
    print(f"\nRaw Response:\n{text}")

    verdict = "Unverified"
    evidence = "Could not parse response."

    if "VERDICT:" in text:
        verdict = text.split("VERDICT:")[1].split("\n")[0].strip()

    if "EVIDENCE:" in text:
        evidence = text.split("EVIDENCE:")[1].strip()

    print(f"\nParsed Verdict: {verdict}")
    print(f"Evidence: {evidence}")

    return {
        "claim": claim,
        "verdict": verdict,
        "evidence": evidence
    }



In [ ]:
def influencer_agent(state: AgentState):
    
    claim = state["claim"]
    verdict = state["verdict"]
    
    print(f"Claim: {claim}")
    print(f"Verdict received: {verdict}")

    if verdict == "Fake":
        prompt = f"Rewrite this claim as a warning post:\n{claim}"
        action_score = 6
        print("Action: Rewriting as WARNING post")
    elif verdict == "Real":
        prompt = f"Rewrite this claim to maximize virality while staying factual:\n{claim}"
        action_score = 8
        print("Action: Rewriting for VIRALITY")
    else:
        prompt = f"Rewrite this claim adding uncertainty and caution:\n{claim}"
        action_score = 4
        print("Action: Rewriting with CAUTION")

    response = generate_content(prompt)
    
    print(f"\nInfluencer Content: {response}")
    print(f"Amplification Score: {action_score}")

    return {
        "claim": claim,
        "verdict": verdict,
        "evidence": state.get("evidence", ""),
        "influencer_content": response,
        "amplification_score": action_score
    }



In [ ]:
def moderator_agent(state: AgentState):
    claim = state["claim"]
    verdict = state["verdict"]
    print(f"Original Claim: {claim}")
    print(f"Verdict received: {verdict}")

    if verdict == "Fake":
        decision = "Flag & Stop"
        print("Decision: FLAG & STOP - Content blocked")
    elif verdict == "Unverified":
        decision = "Mark for Review"
        print("Decision: MARK FOR REVIEW - Needs human review")
    else:
        decision = "Allow"
        print("Decision:  ALLOW - Content approved")


    return {
        "claim": claim,
        "verdict": verdict,
        "evidence": state.get("evidence", ""),
        "influencer_content": state.get("influencer_content", ""),
        "amplification_score": state.get("amplification_score", 0),
        "moderation_decision": decision
    }



In [ ]:
builder = StateGraph(AgentState)

builder.add_node("misinformation", misinformation_agent)
builder.add_node("neutral", neutral_agent)
builder.add_node("fact_check", fact_checker_agent)
builder.add_node("influencer", influencer_agent)
builder.add_node("moderator", moderator_agent)

builder.set_entry_point("misinformation")

builder.add_edge("misinformation", "neutral")
builder.add_edge("neutral", "fact_check")
builder.add_edge("fact_check", "influencer")
builder.add_edge("influencer", "moderator")
builder.add_edge("moderator", END)

graph = builder.compile()


In [ ]:

result = graph.invoke({})
print(json.dumps(result, indent=2))
